In [1]:
#Importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import aux_functions as fun

## 1. Read the data

In [2]:
#Reading data from a file
dF = pd.read_csv("../data/SpotifyFeatures.csv", index_col = None)

In [ ]:
#Visualize
dF

In [ ]:
dF['key'].unique()

In [ ]:
dF['time_signature'].unique()

In [ ]:
dF[dF['popularity']> 56]

In [ ]:
dF.info()

In [ ]:
dF.describe()

## 2. Drop null and duplicated values 

In [ ]:
# Check for missing values
print("Missing values before removal:")
print(dF.isnull().sum())

# Remove rows with missing values
df = dF.dropna()

# Check for missing values after removal
print("\nMissing values after row removal:")
print(df.isnull().sum())

# Print the cleaned DataFrame
print("\nCleaned DataFrame (rows removed):")
df

In [ ]:
dF.duplicated().sum()

## 3. Transform the data

#### 3.1 Transform popularity in nominal data

In [ ]:
#Transforming the popularity of the dataframe into a categorical variable
df['popularity_class'] = pd.cut(df['popularity'], bins = [-float('inf'), 25, 50, 75, 90, float('inf')], labels = ['Very Low','Low','Medium','High','Top'])

In [ ]:
#Checking if the transformation was successsful
df['popularity_class'].unique()

### 3.2 Transform duration in seconds

In [ ]:
#Transforming the duration of the dataframe from milliseconds to seconds
df['duration_s']=df['duration_ms'] / 1000

In [14]:
# Delate columns duration_ms
df = df.drop(['duration_ms'], axis=1)

In [ ]:
df

## 4. Correlation between numerical variables

In [16]:
df_numerical_col = df.select_dtypes(include=['int64', 'float64'])

In [ ]:
#corr=np.abs(df_numerical_col.corr())
corr=df_numerical_col.corr()
#Set up mask for triangle representation
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(10, 10))
# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)
# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, mask=mask,  vmax=1,square=True, linewidths=.5, cbar_kws={"shrink": .5},annot = corr)
plt.title('Correlation matrix')
plt.show()

We can suppose that duration_s is not relevant for the predict model

In [ ]:
for column in df_numerical_col.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x=column, y='popularity', color = 'green', alpha=0.5)
    plt.title(f'Scatter Plot of {column} vs. Popularity')
    plt.xlabel(column)
    plt.ylabel('Popularity')
    plt.show()

## 5. EDA

In [ ]:
df.columns

### 5.1 Genre

In [ ]:
df['genre'].unique()

In [21]:
df['genre'] = df['genre'].replace({"Children's Music":'Children’s Music'})

In [ ]:
df['genre'].nunique()

In [ ]:
frequency_table, proportion_table = fun.create_frequency_table(df,'genre')

In [ ]:
frequency_table

In [ ]:
plt.figure(figsize=(20,30))
frequency_table.plot.pie(autopct='%1.1f%%', startangle=90, colors=sns.color_palette("Set3"))
plt.title('Gender in control')

We can observe the genres are proportionals in the dataset

In [ ]:
df['genre'].index

In [ ]:
plt.figure(figsize=(10,8))
sns.barplot(x=frequency_table.values, y =frequency_table.index, palette='viridis')
plt.xlabel('Count')
plt.ylabel('Genre')
plt.xticks(rotation=55)
plt.title('Count of Songs by Genre')

### 5.1.1 Genre vs Popularity

In [ ]:
df[['genre', 'popularity']]

In [ ]:
df.groupby('genre')['popularity'].mean().sort_values(ascending=False)

In [ ]:
df.groupby("genre").agg(
    mean_popularity=("popularity", "mean"),
    top_artist=("artist_name", lambda x: x.mode()[0])  # Artista más frecuente
).sort_values(by="mean_popularity", ascending=False)

#### 5.1.2 Energy vs Genre

In [ ]:
df.groupby("genre").agg(
    mean_popularity=("popularity", "mean"),
    mean_energy=("energy", "mean"),
    mean_danceability=("danceability", "mean"),
    mean_liveness = ("liveness", "mean"),
    mean_instrumentalness = ("instrumentalness", "mean"),
    mean_loudness = ("loudness", "mean"),
).sort_values(by="mean_energy", ascending=False)

### 5.2 Artist

In [ ]:
df['artist_name'].unique()

In [ ]:
df['artist_name'].nunique()

In [32]:
frequency_table, proportion_table = fun.create_frequency_table(df,'artist_name')

In [ ]:
frequency_table.head(30)

In [ ]:
top_10 = frequency_table.head(10)
top_10

In [ ]:
# Plotting using Seaborn
plt.figure(figsize=(10, 6))
sns.barplot(x=top_10.values, y=top_10.index, palette='Paired')
plt.xlabel('Count')
plt.ylabel('Artist Name')
plt.title('Top 10 Artists by Number of Songs')
plt.show()

In [ ]:
artist_popu = df.groupby('artist_name').agg(
    mean_popularity=("popularity", "mean"),
    genre=("genre", "first")).sort_values(by = 'mean_popularity',ascending=False).reset_index()
artist_popu

In [ ]:
artist_popu.head(30)

## 6. Outliers

In [ ]:
for column in df_numerical_col.columns:
    plt.figure(figsize=(4, 2))
    sns.boxplot(y=column, data=df_numerical_col)
    plt.title(f'Box plot of {column}')
    plt.xlabel(column)
    plt.show()

In [ ]:
num_columns = len(df_numerical_col.columns)  # Número de columnas numéricas
rows = (num_columns + 1) // 2  # Si hay un número impar de columnas, se añadirá una fila extra
cols = 2  # Número de columnas por fila

# Crear los subgráficos
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(cols * 4, rows * 4), constrained_layout=True)

# Aplanamos el array de ejes para iterar más fácilmente
axes = axes.flatten()

# Generar un gráfico para cada columna
for i, column in enumerate(df_numerical_col.columns):
    sns.boxplot(y=column, data=df_numerical_col, ax=axes[i])
    axes[i].set_title(f'Box plot of {column}')
    axes[i].set_xlabel(column)

# Eliminar subgráficos vacíos si el número de columnas es impar
for j in range(len(df_numerical_col.columns), len(axes)):
    axes[j].remove()

# Mostrar la figura
plt.show()

In [ ]:
for column in df_numerical_col.columns:
    fun.percentage_outliers(df_numerical_col, column)

## Check the popularity balance

In [ ]:
df['popularity_class'].value_counts()

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Separar características (X) y target (y)
X = df.drop(columns=['popularity_class'])  # Variables independientes
y = df['popularity_class']  # Variable dependiente

# Aplicar SMOTE primero
smote = SMOTE(sampling_strategy={"High": 10000, "Top": 5000}, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Aplicar submuestreo después
under = RandomUnderSampler(sampling_strategy={"Low": 50000, "Medium": 50000}, random_state=42)
X_resampled, y_resampled = under.fit_resample(X_resampled, y_resampled)

# Crear nuevo DataFrame balanceado
df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_resampled['popularity_class'] = y_resampled

# Verificar balance final
print(df_resampled['popularity_class'].value_counts())


## Data set without popularity class

In [104]:
df_reg = df.drop(columns='popularity_class')
df_reg.columns
df_reg.to_csv('../data/spotify_clean_reg.csv')

In [ ]:
df_reg.columns

In [106]:
df_numerical_reg = df_reg.select_dtypes(include=['int64', 'float64'])

In [ ]:
# Crear un DataFrame acumulativo eliminando los outliers
df_reg_without_outliers = df_numerical_reg.copy()

for column in df_numerical_reg.columns:
    df_reg_without_outliers = fun.remove_outliers(df_reg_without_outliers, column)

# Calcular el porcentaje de outliers para las columnas resultantes
for column in df_reg_without_outliers.columns:
    fun.percentage_outliers(df_reg_without_outliers, column)


In [ ]:
df_reg_without_outliers

In [ ]:
num_columns = len(df_reg_without_outliers.columns)  # Número de columnas numéricas
rows = (num_columns + 1) // 2  # Si hay un número impar de columnas, se añadirá una fila extra
cols = 2  # Número de columnas por fila

# Crear los subgráficos
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(cols * 4, rows * 4), constrained_layout=True)

# Aplanamos el array de ejes para iterar más fácilmente
axes = axes.flatten()

# Generar un gráfico para cada columna
for i, column in enumerate(df_reg_without_outliers.columns):
    sns.boxplot(y=column, data=df_reg_without_outliers, ax=axes[i])
    axes[i].set_title(f'Box plot of {column}')
    axes[i].set_xlabel(column)

# Eliminar subgráficos vacíos si el número de columnas es impar
for j in range(len(df_reg_without_outliers.columns), len(axes)):
    axes[j].remove()

# Mostrar la figura
plt.show()

In [114]:
df_reg_outliers_final = pd.concat([df_reg.drop(columns=df_numerical_reg.columns), df_reg_without_outliers], axis=1)


In [ ]:
df_reg_outliers_final.columns

In [116]:
df_reg_outliers_final.to_csv('../data/spotify_clean_reg_without_outliers.csv')

## Save the clean data

In [44]:
df.to_csv('../data/spotify_clean.csv')